In [2]:
#Assignment 1 — Patient Clinical Data Analysis
import pandas as pd

#Part A — Understanding the Dataset

dataset = pd.read_csv("patient_clinical_data_raw.csv")

df = pd.DataFrame(dataset) 
df.head(10) #First 10 records
df.tail(10) #Last 10 records
df.shape #Find the number of rows and columns.
df.info() #Display the column names and their data types.
df.describe() #Generate a statistical summary of the numerical columns.

#Identify the number of unique values in each column.
unique_counts = df.nunique()

print("Number of unique values in each column:")
print(unique_counts)



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10100 entries, 0 to 10099
Data columns (total 14 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Patient_ID      10100 non-null  object 
 1   Age             9798 non-null   float64
 2   Gender          10100 non-null  object 
 3   Department      10100 non-null  object 
 4   Blood_Pressure  10100 non-null  object 
 5   Heart_Rate      9798 non-null   float64
 6   Temperature     9796 non-null   float64
 7   Cholesterol     9797 non-null   float64
 8   Glucose         9798 non-null   float64
 9   Diagnosis       10100 non-null  object 
 10  Admission_Type  10100 non-null  object 
 11  Length_of_Stay  10100 non-null  int64  
 12  Treatment_Cost  10100 non-null  float64
 13  Readmission     10100 non-null  object 
dtypes: float64(6), int64(1), object(7)
memory usage: 1.1+ MB
Number of unique values in each column:
Patient_ID        10000
Age                  92
Gender               

In [43]:
#Part B — Data Cleaning
import numpy as np

#7. Identify all columns containing missing values.
missing_columns = df.columns[df.isnull().any()].tolist()
print(missing_columns)

#8. Calculate the percentage of missing values in each column.
missing_percentage = df.isnull().mean() * 100
print(missing_percentage.round(0).astype(int))

#9. Handle missing values in the numerical columns appropriately.
num_cols = df.select_dtypes(include=[np.number]).columns # Get names of all numerical columns
df[num_cols] = df[num_cols].fillna(0)

#10. Identify and remove duplicate records.
print(df[df.duplicated(keep=False)])
df.drop_duplicates(inplace=True)
df.shape

#11. Check the Age column for invalid values.
invalid_ages = df[(df['Age'] < 0) | (df['Age'] > 110)]
print(invalid_ages)

#12. Correct or handle patients whose age is outside a reasonable range.
df['Age'] = df['Age'].clip(lower=0, upper=110)

#13. Identify inconsistent values in categorical columns such as Gender and Department.
## Check for spelling, case, or formatting inconsistencies
print(df['Gender'].value_counts(dropna=False))
print(df['Department'].value_counts(dropna=False))

#14. Standardize these categorical values.
df['Department'] = df['Department'].astype(str).str.strip().str.title()
df['Gender'] = df['Gender'].astype(str).str.strip().str.title()
print(df['Gender'].value_counts(dropna=False))
print(df['Department'].value_counts(dropna=False))


[]
Patient_ID        0
Age               0
Gender            0
Department        0
Blood_Pressure    0
Heart_Rate        0
Temperature       0
Cholesterol       0
Glucose           0
Diagnosis         0
Admission_Type    0
Length_of_Stay    0
Treatment_Cost    0
Readmission       0
dtype: int64
Empty DataFrame
Columns: [Patient_ID, Age, Gender, Department, Blood_Pressure, Heart_Rate, Temperature, Cholesterol, Glucose, Diagnosis, Admission_Type, Length_of_Stay, Treatment_Cost, Readmission]
Index: []
Empty DataFrame
Columns: [Patient_ID, Age, Gender, Department, Blood_Pressure, Heart_Rate, Temperature, Cholesterol, Glucose, Diagnosis, Admission_Type, Length_of_Stay, Treatment_Cost, Readmission]
Index: []
Gender
Male        4920
Female      4780
MALE         100
male         100
 FEMALE      100
Name: count, dtype: int64
Department
Oncology            1662
Orthopedics         1644
Pediatrics          1614
Neurology           1612
General Medicine    1607
Cardiology          1561
 CARDIOLO

In [57]:
#Part C — Data Transformation
#15. Create a new column called Age_Group:
#< 18 Child,18–40 Young Adult, 41–60 Middle Age,> 60 Senior


# Define the boundaries for the bins
# -np.inf covers negative numbers/0, and np.inf covers all ages above 60
bins = [-np.inf, 17, 40, 60, np.inf]

# Define the matching labels for each bin range
labels = ['Child', 'Young Adult', 'Middle Age', 'Senior']

# Create the new column
df['Age_Group'] = pd.cut(df['Age'], bins=bins, labels=labels)
print(df['Age_Group'])
df.to_csv('patient_clinical_data_raw1.csv', index=False)

#16. Separate Blood_Pressure into two columns:
#• Systolic
#• Diastolic
## Split by the slash '/' and expand the results into two new columns
df[['Systolic', 'Diastolic']] = df['Blood_Pressure'].astype(str).str.split('/', expand=True)
# Convert to integer, converting any broken values (like text) into NaN
df['Systolic'] = pd.to_numeric(df['Systolic'], errors='coerce')
df['Diastolic'] = pd.to_numeric(df['Diastolic'], errors='coerce')

#17. Create a High_Risk column based on suitable clinical conditions.
# List of clinical conditions
conditions = [
    (df['Systolic'] >= 140) | (df['Diastolic'] >= 90),           # Severe BP
    (df['Age_Group'] == 'Senior') & (df['Systolic'] >= 130),     # Vulnerable Senior
    (df['Systolic'].isna()) | (df['Diastolic'].isna())           # Missing data
]

# Matching labels for each condition
choices = ['High Risk', 'High Risk', 'Unknown']

# Assign labels, default to 'Normal Risk' if no conditions are met
df['High_Risk'] = np.select(conditions, choices, default='Normal Risk')


#18. Create a Cost_Per_Day column using treatment cost and length of stay.
# Convert to numeric format safely
df['Treatment_Cost'] = pd.to_numeric(df['Treatment_Cost'], errors='coerce')
df['Length_of_Stay'] = pd.to_numeric(df['Length_of_Stay'], errors='coerce')

df['Cost_Per_Day'] = (df['Treatment_Cost']/df['Length_of_Stay']).round(2)
#df.to_csv('patient_clinical_data_raw2.csv', index=False)

0         Middle Age
1              Child
2         Middle Age
3              Child
4         Middle Age
            ...     
10095    Young Adult
10096     Middle Age
10097    Young Adult
10098     Middle Age
10099          Child
Name: Age_Group, Length: 10000, dtype: category
Categories (4, object): ['Child' < 'Young Adult' < 'Middle Age' < 'Senior']


In [11]:
#Part D — Analysis
#19. Calculate the average age of patients.
avg_age = df['Age'].mean()
print(avg_age)

#20. Find the number of patients in each department.
# Count the number of patients per department
department_counts = df['Department'].value_counts()
print(department_counts)

#21. Calculate the average treatment cost for each department.
# Group by department and calculate the mean cost
avg_cost_by_dept = df.groupby('Department')['Treatment_Cost'].mean().round(2)
print(avg_cost_by_dept)

#22. Find the average length of stay for each diagnosis.
avg_length_of_stay = df.groupby('Diagnosis')['Length_of_Stay'].mean().round(2)
print(avg_length_of_stay)

#23. Identify the top 10 patients with the highest treatment cost.
## Extract the top 10 rows with the highest treatment costs
top_10_patients = df.nlargest(10, 'Treatment_Cost')
print(top_10_patients)

#24. Find the department with the highest average treatment cost.
avg_costs = df.groupby('Department')['Treatment_Cost'].mean()
# 2. Find the name of the department with the maximum average cost
highest_dept = avg_costs.idxmax()
print('Department with highest avg cost',highest_dept)

#25. Calculate the percentage of patients who were readmitted.
readmission_percentage = (df['Readmission'] == 'Yes').mean() * 100
print('readmission_percentage', readmission_percentage)

#26. Compare readmission rates across departments.
# Group by department and calculate the percentage of 'Yes' values
dept_readmission_rates = df.groupby('Department')['Readmission'].apply(lambda x: (x == 'Yes').mean() * 100)
print('dept_readmission_rates', dept_readmission_rates)

# Sort from highest readmission rate to lowest
dept_readmission_rates = dept_readmission_rates.sort_values(ascending=False).round(2)
print('Sorted Value', dept_readmission_rates)


#27. Find the average cholesterol and glucose level by diagnosis.
print('cholesterol average :' , df['Cholesterol'].mean().round(2))
print('Glucose average:', df['Glucose'].mean().round(2));

#28. Identify patients with:
#• high glucose
print('Patient with highest glucose :' , df.loc[df['Glucose'].nlargest(1).idxmax()])
#• high cholesterol
print('Patient with highest cholesterol :' , df.loc[df['Cholesterol'].nlargest(1).idxmax()])
#• high heart rate
print('Patient with highest heart rate :' , df.loc[df['Heart_Rate'].nlargest(1).idxmax()])

47.41824862216779
Department
Oncology            1681
Orthopedics         1657
Pediatrics          1629
Neurology           1625
General Medicine    1624
Cardiology          1582
neurology            102
 CARDIOLOGY          100
cardiology           100
Name: count, dtype: int64
Department
 CARDIOLOGY         104652.10
Cardiology          101950.70
General Medicine    100693.31
Neurology           101642.44
Oncology             99916.25
Orthopedics         100221.42
Pediatrics           98918.93
cardiology          102687.64
neurology            93707.16
Name: Treatment_Cost, dtype: float64
Diagnosis
Asthma           4.99
Cancer           5.00
Diabetes         5.08
Fracture         5.04
Heart Disease    5.00
Hypertension     5.10
Infection        5.02
Migraine         5.05
Name: Length_of_Stay, dtype: float64
      Patient_ID   Age  Gender   Department Blood_Pressure  Heart_Rate  \
1796      P09106  41.0    Male    Neurology         117/73        88.0   
10066     P01900  42.0  Female 